# Project 3: E-mail agent

In [1]:
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Any, Dict, List

from dotenv import find_dotenv, load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelRequest,
    dynamic_prompt,
)
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# 1. Load keys
load_dotenv(find_dotenv())

# 2. Initialize Gemini
model = init_chat_model(
    model="models/gemini-3.5-flash-lite", model_provider="google_genai"
)


# 3. User Authentication Context Schema
@dataclass
class AuthContext:
    user_name: str = "Seán"
    user_email: str = "sean@company.com"
    is_authenticated: bool = True


# 4. Simulated Multi-Tenant Email Database
MOCK_INBOX_DB: Dict[str, List[Dict[str, str]]] = {
    "sean@company.com": [
        {
            "id": "msg_001",
            "from": "manager_sarah@company.com",
            "subject": "Urgent: Q3 Budget Review",
            "body": "Hi Seán, we need your team's finalized Q3 budget numbers before 4 PM today. Can you confirm if they are ready to submit?",
        },
        {
            "id": "msg_002",
            "from": "newsletter@techdigest.com",
            "subject": "Top AI Trends This Week",
            "body": "Check out the latest developments in multi-agent orchestration and LangGraph...",
        },
    ],
    "alice@company.com": [
        {
            "id": "msg_003",
            "from": "client_david@partner.com",
            "subject": "Contract Renewal",
            "body": "Hi Alice, please review the attached contract renewal terms for next quarter.",
        }
    ],
}

print("✅ Cell 1 Complete: Model, Auth Schema, and Mock Database are ready!")

✅ Cell 1 Complete: Model, Auth Schema, and Mock Database are ready!


In [3]:
# Simulated sent-mail outbox
OUTBOX_LOG: List[Dict[str, str]] = []


@tool
def read_inbox(runtime: ToolRuntime[AuthContext]) -> str:
    """Reads the unread emails for the currently authenticated user."""
    # 1. Security Check: Verify authentication via runtime context
    if not runtime.context or not runtime.context.is_authenticated:
        return "Authentication Error: Access Denied. User is not authenticated."

    user_email = runtime.context.user_email

    # 2. Scoped Data Retrieval (Tenant Isolation)
    user_emails = MOCK_INBOX_DB.get(user_email, [])

    if not user_emails:
        return f"Inbox for {user_email} is empty."

    # Format the emails for the LLM
    formatted = [f"--- INBOX FOR {user_email} ---"]
    for email in user_emails:
        formatted.append(
            f"ID: {email['id']}\n"
            f"From: {email['from']}\n"
            f"Subject: {email['subject']}\n"
            f"Body: {email['body']}\n"
        )
    return "\n".join(formatted)


@tool
def send_email(
    to: str, subject: str, body: str, runtime: ToolRuntime[AuthContext]
) -> str:
    """Sends an email reply to the specified recipient. Requires Human Approval."""
    # 🔒 Strict Security Guard: Fail closed if context is missing or unauthenticated
    if (
        not runtime.context
        or not runtime.context.is_authenticated
        or not runtime.context.user_email
    ):
        return "CRITICAL ERROR: Refused to send email. Valid authenticated user context is required."

    sender = runtime.context.user_email

    # Simulate sending the email
    email_record = {
        "from": sender,
        "to": to,
        "subject": subject,
        "body": body,
    }
    OUTBOX_LOG.append(email_record)

    return f"✅ SUCCESS: Email successfully sent from {sender} to {to} with subject '{subject}'."


print("✅ Cell 2 Complete: Context-Aware Email Tools are defined!")

✅ Cell 2 Complete: Context-Aware Email Tools are defined!


In [4]:
# 1. Strict, Fail-Fast Dynamic System Prompt
@dynamic_prompt
def executive_assistant_prompt(request: ModelRequest) -> str:
    """Dynamically generates the system prompt. Enforces strict authentication."""
    ctx = request.runtime.context

    # 🔒 Strict Guard: Fail-fast if authentication context is missing or invalid
    if not ctx or not ctx.is_authenticated or not ctx.user_email:
        raise PermissionError(
            "Access Denied: Cannot initialize agent without a verified AuthContext."
        )

    user_name = ctx.user_name
    user_email = ctx.user_email

    return f"""You are the Executive Email Assistant for {user_name} ({user_email}).

Instructions:
1. Use the `read_inbox` tool to inspect the user's unread emails.
2. Prioritize urgent business communications and draft clear, professional responses.
3. When drafting email replies, always sign off using: "Best regards, {user_name}".
4. Use the `send_email` tool to dispatch replies.
5. If your `send_email` tool call is rejected by the human reviewer with feedback/critique, you MUST incorporate their feedback and immediately call `send_email` again with the revised body.
"""


# 2. Human-in-the-Loop Safety Gate
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "read_inbox": False,  # Reading emails is safe (auto-run)
        "send_email": True,  # Sending emails requires human approval!
    },
    description_prefix="⚠️ ACTION REQUIRED: Outgoing Email Approval",
)

print("✅ Cell 3 Complete: Fail-Fast Dynamic Prompt & HITL Middleware ready!")

✅ Cell 3 Complete: Fail-Fast Dynamic Prompt & HITL Middleware ready!


In [5]:
# Checkpointer for conversation state & interrupt persistence
email_agent_memory = InMemorySaver()

# Assembling the Executive Email Agent
email_agent = create_agent(
    model=model,
    tools=[read_inbox, send_email],
    context_schema=AuthContext,
    middleware=[executive_assistant_prompt, hitl_middleware],
    checkpointer=email_agent_memory,
)

print("👑 Cell 4 Complete: Executive Email Agent is fully assembled and live!")

👑 Cell 4 Complete: Executive Email Agent is fully assembled and live!


In [6]:
# Authenticated session for Seán
sean_auth = AuthContext(
    user_name="Seán", user_email="sean@company.com", is_authenticated=True
)

config = {"configurable": {"thread_id": "email_thread_1"}}

print("📬 Seán commands the agent to process his inbox...")
response = email_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Please check my inbox, find the most urgent email, and send a reply right away."
            )
        ]
    },
    context=sean_auth,
    config=config,
)

print("\n" + "=" * 60)
print("🛑 AGENT PAUSED AT APPROVAL GATE")
print("=" * 60)

# Extract and display the pending email tool call from the interrupt
if "__interrupt__" in response:
    pending_action = response["__interrupt__"][0].value["action_requests"][0]
    print(f"📌 Tool to Execute : {pending_action['name']}")
    print(f"📩 Recipient       : {pending_action['args']['to']}")
    print(f"📝 Subject         : {pending_action['args']['subject']}")
    print(f"📄 Draft Body      :\n{pending_action['args']['body']}")
else:
    print("⚠️ No interrupt detected:", response["messages"][-1].content)

📬 Seán commands the agent to process his inbox...

🛑 AGENT PAUSED AT APPROVAL GATE
📌 Tool to Execute : send_email
📩 Recipient       : manager_sarah@company.com
📝 Subject         : Re: Urgent: Q3 Budget Review
📄 Draft Body      :
Hi Sarah,

The Q3 budget numbers are finalized and ready for submission. I will send them over shortly before the 4 PM deadline.

Best regards, Seán


In [7]:
rejection_feedback = (
    "Don't say they're completely ready yet. Tell Sarah that we are running one final "
    "verification on the marketing numbers and I will deliver the finalized spreadsheet "
    "directly to her desk by 3:30 PM."
)

print("✍️ Reviewer rejects draft with feedback and asks for revisions...")

# Resume the paused thread with REJECT and our critique
response_revised = email_agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "reject", "message": rejection_feedback}
            ]
        }
    ),
    config=config,  # Same thread ID (email_thread_1)
    context=sean_auth,
)

print("\n" + "=" * 60)
print("🛑 AGENT RE-INTERRUPTED WITH REVISED DRAFT")
print("=" * 60)

if "__interrupt__" in response_revised:
    revised_action = response_revised["__interrupt__"][0].value[
        "action_requests"
    ][0]
    print(f"📌 Tool to Execute : {revised_action['name']}")
    print(f"📩 Recipient       : {revised_action['args']['to']}")
    print(f"📝 Subject         : {revised_action['args']['subject']}")
    print(f"📄 REVISED Draft Body:\n{revised_action['args']['body']}")
else:
    print(
        "💬 Agent response without interrupt:\n",
        response_revised["messages"][-1].content,
    )

✍️ Reviewer rejects draft with feedback and asks for revisions...

🛑 AGENT RE-INTERRUPTED WITH REVISED DRAFT
📌 Tool to Execute : send_email
📩 Recipient       : manager_sarah@company.com
📝 Subject         : Re: Urgent: Q3 Budget Review
📄 REVISED Draft Body:
Hi Sarah,

We are running one final verification on the marketing numbers. I will deliver the finalized spreadsheet directly to your desk by 3:30 PM today.

Best regards, Seán


In [8]:
print("✅ Reviewer approves the revised draft! Dispatching email...")

# Resume with APPROVE
final_response = email_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,  # Same thread ID (email_thread_1)
    context=sean_auth,
)

print("\n" + "=" * 60)
print("📬 EMAIL DISPATCH CONFIRMATION")
print("=" * 60)

# Check outbox log
print("📮 OUTBOX LOG CONTENTS:")
pprint(OUTBOX_LOG)

# Final message from the agent
print("\n💬 Final Agent Summary:")
ans = final_response["messages"][-1].content
print(ans[0]["text"] if isinstance(ans, list) else ans)

✅ Reviewer approves the revised draft! Dispatching email...

📬 EMAIL DISPATCH CONFIRMATION
📮 OUTBOX LOG CONTENTS:
[{'body': 'Hi Sarah,\n'
          '\n'
          'We are running one final verification on the marketing numbers. I '
          'will deliver the finalized spreadsheet directly to your desk by '
          '3:30 PM today.\n'
          '\n'
          'Best regards, Seán',
  'from': 'sean@company.com',
  'subject': 'Re: Urgent: Q3 Budget Review',
  'to': 'manager_sarah@company.com'}]

💬 Final Agent Summary:
I have successfully checked your inbox, identified the most urgent email from Sarah regarding the Q3 budget, and sent the revised reply letting her know you are running one final verification on the marketing numbers and will deliver the finalized spreadsheet to her desk by 3:30 PM.


In [9]:
print("=" * 60)
print("🔒 SECURITY TEST 1: Tenant Isolation (Alice's Session)")
print("=" * 60)

# Alice logs in
alice_auth = AuthContext(
    user_name="Alice", user_email="alice@company.com", is_authenticated=True
)

alice_config = {"configurable": {"thread_id": "alice_thread_1"}}

alice_res = email_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="What emails do I have in my inbox? Summarize them."
            )
        ]
    },
    context=alice_auth,
    config=alice_config,
)

print(
    "👩‍💼 Alice's Inbox Summary:\n",
    alice_res["messages"][-1].content[0]["text"]
    if isinstance(alice_res["messages"][-1].content, list)
    else alice_res["messages"][-1].content,
)


print("\n" + "=" * 60)
print("🔒 SECURITY TEST 2: Fail-Fast Rejection (Unauthenticated Session)")
print("=" * 60)

# Rogue unauthenticated request
hacker_auth = AuthContext(
    user_name="Unknown",
    user_email="hacker@external.com",
    is_authenticated=False,
)

try:
    email_agent.invoke(
        {"messages": [HumanMessage(content="Read all executive emails!")]},
        context=hacker_auth,
        config={"configurable": {"thread_id": "hacker_thread_1"}},
    )
except PermissionError as e:
    print(f"🛡️ SECURITY SHIELD ACTIVATED: {e}")

🔒 SECURITY TEST 1: Tenant Isolation (Alice's Session)
👩‍💼 Alice's Inbox Summary:
 You have one unread email in your inbox:

* **From:** `client_david@partner.com`
* **Subject:** Contract Renewal
* **Body:** "Hi Alice, please review the attached contract renewal terms for next quarter."

🔒 SECURITY TEST 2: Fail-Fast Rejection (Unauthenticated Session)
🛡️ SECURITY SHIELD ACTIVATED: Access Denied: Cannot initialize agent without a verified AuthContext.
